# 40_model_selection_main
**NOVA IMS – DSAA 2025/26**  
**Cars 4 You – Model Selection, Final Training & Submission**  
*Generated on:* 2025-10-17

Dieses Notebook baut auf `30_feature_selection_main` auf:
- lädt das **finale Feature-Set** aus `artifacts/selected_features.json`
- führt **Nested-CV-Model-Selection** (MAE primär) über mehrere sklearn-Modelle durch
- trainiert das **beste Modell** auf dem Gesamtdatensatz
- erzeugt **Vorhersagen** für `test.csv` und exportiert `submissions/submission.csv`.

## Setup & Imports

In [20]:
# ===== Projektpfade (auto-detected; bei Bedarf anpassen) =====
from pathlib import Path

DATA_DIR  = Path("../data")
TRAIN_FILE = DATA_DIR / "processed_train_data.csv"
TEST_FILE  = DATA_DIR / "test.csv"
TARGET = "price"

ARTIFACTS = Path("artifacts")
SUBMISSIONS = Path("submissions"); SUBMISSIONS.mkdir(exist_ok=True, parents=True)

print("DATA_DIR :", DATA_DIR)
print("TRAIN   :", TRAIN_FILE)
print("TEST    :", TEST_FILE)
print("TARGET  :", TARGET)

DATA_DIR : ../data
TRAIN   : ../data/processed_train_data.csv
TEST    : ../data/test.csv
TARGET  : price


In [21]:
# ================
# Imports
# ================
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.base import clone

## Daten & Preprocessing

In [22]:
# =======================
# Daten laden
# =======================
train = pd.read_csv(TRAIN_FILE)
test  = pd.read_csv(TEST_FILE)

assert TARGET in train.columns, f"Zielvariable '{TARGET}' nicht gefunden!"

X = train.drop(columns=[TARGET])
y = train[TARGET].copy()

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print("Train:", train.shape, "| Test:", test.shape, "| num:", len(num_cols), "| cat:", len(cat_cols))

Train: (75973, 14) | Test: (32567, 13) | num: 9 | cat: 4


In [23]:
# =============================================
# Preprocessing (aligned mit 20/30)
# =============================================
numeric_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler(with_mean=True, with_std=True)),
])

categorical_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

## Finale Featuremenge laden

In [24]:
# =============================================
# Finale Featuremenge aus 30er Notebook laden
# =============================================
SEL_FILE = ARTIFACTS / "selected_features.json"
assert SEL_FILE.exists(), f"Feature-Selektionsdatei nicht gefunden: {SEL_FILE.resolve()}"

meta = json.loads(SEL_FILE.read_text(encoding="utf-8"))
final_features = set(meta["final_features"])
print("Geladene finale Features:", len(final_features))

# Fit preprocessor to get transformed feature names
_ = preprocessor.fit(X)

def get_feature_names(preprocessor, num_cols, cat_cols):
    feature_names = []
    feature_names += [f"NUM::{c}" for c in num_cols]
    ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
    ohe_names = list(ohe.get_feature_names_out(cat_cols))
    feature_names += [f"CAT::{n}" for n in ohe_names]
    return feature_names

feature_names = get_feature_names(preprocessor, num_cols, cat_cols)
name_to_idx = {n:i for i, n in enumerate(feature_names)}

keep_idx = [name_to_idx[n] for n in feature_names if n in final_features]

def mask_final(X_):
    Xt = preprocessor.transform(X_)
    return Xt[:, keep_idx]

Geladene finale Features: 60


## CV-Setup & Kandidaten

In [25]:
# ==================================
# CV & Scorer
# ==================================
RANDOM_STATE = 42
OUTER_FOLDS = 10
INNER_FOLDS = 5
N_JOBS = -1

kf_outer = KFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
kf_inner = KFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

scorer_mae  = make_scorer(mean_absolute_error, greater_is_better=False)
scorer_rmse = make_scorer(lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)), greater_is_better=False)
scorer_r2   = make_scorer(r2_score, greater_is_better=True)

In [26]:
# ==================================
# Model-Kandidaten & Grids
# ==================================
models_and_grids = {
    "LinearRegression": (LinearRegression(), {}),
    "Ridge": (Ridge(random_state=RANDOM_STATE), {"alpha": [0.1, 1.0, 3.0, 10.0, 30.0]}),
    "Lasso": (Lasso(random_state=RANDOM_STATE, max_iter=20000), {"alpha": [0.0005, 0.001, 0.01, 0.1, 1.0]}),
    "ElasticNet": (ElasticNet(random_state=RANDOM_STATE, max_iter=20000), {
        "alpha": [0.0005, 0.001, 0.01, 0.1, 1.0],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
    }),
    "RandomForest": (RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS), {
        "n_estimators": [300, 600],
        "max_depth": [None, 12, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }),
    "GradientBoosting": (GradientBoostingRegressor(random_state=RANDOM_STATE), {
        "n_estimators": [300, 600],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [2, 3],
        "subsample": [0.8, 1.0]
    }),
    "SVR": (SVR(), {
        "C": [1.0, 3.0, 10.0],
        "epsilon": [0.01, 0.05, 0.1],
        "kernel": ["rbf"],
        "gamma": ["scale"]
    }),
}

## Nested-CV Model Selection

In [27]:
# --- Recreate 'final' if not imported from previous notebook ---
if "final" not in globals():
    print("⚠️  Kein 'final' Feature-Set gefunden – lade aus artifacts/selected_features.json ...")
    import json
    from pathlib import Path

    SEL_FILE = Path("artifacts/selected_features.json")
    assert SEL_FILE.exists(), "selected_features.json fehlt – bitte zuerst Notebook 30 ausführen!"
    meta = json.loads(SEL_FILE.read_text(encoding="utf-8"))
    final = set(meta["final_features"])
    print(f"→ {len(final)} finale Features geladen.")


In [28]:
# ============================================================
# NESTED CV (SMOKE TEST) – minimal & schnell
# ============================================================
import time, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from scipy.stats import loguniform

t0 = time.time()

# ---------- 0) 'final' Feature-Set ggf. laden ----------
if "final" not in globals():
    SEL_FILE = Path("artifacts/selected_features.json")
    assert SEL_FILE.exists(), "selected_features.json fehlt – erst Notebook 30 laufen lassen!"
    import json
    final = set(json.loads(SEL_FILE.read_text(encoding="utf-8"))["final_features"])

# ---------- 1) Einmalig transformieren & Final-Maske bilden ----------
_ = preprocessor.fit(X)
X_t = preprocessor.transform(X)

# feature_names ggf. rekonstruieren (NUM::..., CAT::...)
if "feature_names" not in globals():
    ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
    feature_names = [f"NUM::{c}" for c in num_cols] + [f"CAT::{n}" for n in ohe.get_feature_names_out(cat_cols)]

idx_final = np.array([i for i, n in enumerate(feature_names) if n in final])
X_final = np.ascontiguousarray(X_t[:, idx_final])
y_arr = y.values if hasattr(y, "values") else np.asarray(y)

# ---------- 2) Schnelle Splits ----------
OUTER_FOLDS = 3   # klein halten
INNER_FOLDS = 2
outer = KFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=42)

# ---------- 3) Modelle: nur minimale Auswahl ----------
RUN_ONLY = ["LinearRegression", "RandomForest"]  # -> bei Bedarf anpassen

models_and_dists = {
    "LinearRegression": (LinearRegression(), None),
    "RandomForest": (RandomForestRegressor(random_state=42, n_jobs=-1), {
        "n_estimators": [150, 250],           # sehr kleiner Raum
        "max_depth": [None, 12],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }),
    # Optional weitere hinzufügen:
    # "Ridge": (Ridge(random_state=42), {"alpha": loguniform(1e-2, 1e2)})
}

N_ITER = 5  # sehr kleine Randomized-Suche

results_smoke = []
for name in RUN_ONLY:
    est, dist = models_and_dists[name]
    maes = []
    for tr_idx, te_idx in outer.split(X_final, y_arr):
        Xtr, Xte = X_final[tr_idx], X_final[te_idx]
        ytr, yte = y_arr[tr_idx], y_arr[te_idx]

        if dist is None:
            mdl = clone(est).fit(Xtr, ytr)
        else:
            rs = RandomizedSearchCV(
                clone(est), dist, n_iter=N_ITER, cv=INNER_FOLDS,
                scoring="neg_mean_absolute_error", random_state=42,
                n_jobs=-1, refit=True, error_score="raise"
            )
            rs.fit(Xtr, ytr)
            mdl = rs.best_estimator_

        pred = mdl.predict(Xte)
        maes.append(mean_absolute_error(yte, pred))

    mean_mae, std_mae = float(np.mean(maes)), float(np.std(maes))
    results_smoke.append((name, mean_mae, std_mae))
    print(f"{name:>16} | MAE: {mean_mae:.4f} ± {std_mae:.4f}  (outer={OUTER_FOLDS}, inner={INNER_FOLDS})")

res_df_fast = pd.DataFrame(results_smoke, columns=["model", "mae_mean", "mae_std"]).sort_values("mae_mean")
print(f"\nSMOKE DONE in {time.time()-t0:.1f}s")
res_df_fast


LinearRegression | MAE: 2839.6367 ± 12.0997  (outer=3, inner=2)
    RandomForest | MAE: 1497.4510 ± 9.9583  (outer=3, inner=2)

SMOKE DONE in 165.1s


,model,mae_mean,mae_std
1,RandomForest,1497.450962,9.958331
0,LinearRegression,2839.636676,12.099710


## Finales Training & OOF-Evaluation

In [29]:
# ==================================
# FAST: Bestes Modell refitten + OOF
# ==================================
import time, math, numpy as np
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor

t0 = time.time()

# --- 0) Sicherstellen, dass wir im transformierten Final-Feature-Raum sind ---
if "X_final" not in globals():
    _ = preprocessor.fit(X)
    X_t = preprocessor.transform(X)
    if "feature_names" not in globals():
        ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
        feature_names = [f"NUM::{c}" for c in num_cols] + [f"CAT::{n}" for n in ohe.get_feature_names_out(cat_cols)]
    idx_final = np.array([i for i, n in enumerate(feature_names) if n in final])
    X_final = np.ascontiguousarray(X_t[:, idx_final])
y_arr = y.values if hasattr(y, "values") else np.asarray(y)

# --- 1) Bestes Modell aus res_df_fast holen (aus der schnellen Nested-CV) ---
best_row = res_df_fast.iloc[0]
BEST_MODEL = best_row["model"]
print("Best model:", BEST_MODEL, "| MAE nested:", best_row["mae_mean"])

# Fallback: falls models_and_grids nicht definiert ist, minimal definieren
if "models_and_grids" not in globals():
    from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.svm import SVR
    models_and_grids = {
        "LinearRegression": (LinearRegression(), {}),
        "Ridge": (Ridge(random_state=42), {"alpha": [0.1, 1.0, 10.0, 30.0]}),
        "Lasso": (Lasso(random_state=42, max_iter=20000), {"alpha": [0.0005, 0.001, 0.01, 0.1]}),
        "ElasticNet": (ElasticNet(random_state=42, max_iter=20000), {
            "alpha": [0.0005, 0.001, 0.01, 0.1],
            "l1_ratio": [0.2, 0.5, 0.8]
        }),
        "RandomForest": (RandomForestRegressor(random_state=42, n_jobs=-1), {
            "n_estimators": [200, 350, 500],
            "max_depth": [None, 12, 20],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2],
        }),
        "GradientBoosting": (GradientBoostingRegressor(random_state=42), {
            "n_estimators": [200, 400],
            "learning_rate": [0.05, 0.1],
            "max_depth": [2, 3],
        }),
        "SVR": (SVR(), {
            "C": [1.0, 3.0, 10.0],
            "epsilon": [0.01, 0.1],
            "kernel": ["rbf"],
            "gamma": ["scale"]
        }),
    }

estimator, grid = models_and_grids[BEST_MODEL]

# --- 2) Kleines, schnelles Tuning (falls Grid vorhanden) direkt auf Arrays ---
def small_random_search(est, grid, Xf, yf, inner_folds=3, n_iter=10):
    # RandomizedSearch darf auch Listen verwenden; n_iter auf sinnvolles Minimum kappen
    # (z.B. <= Anzahl Kombis)
    def count_combos(g):
        total = 1
        for v in g.values():
            total *= len(v)
        return total
    max_iter = min(n_iter, count_combos(grid)) if grid else 0
    if not grid or max_iter == 0:
        return clone(est).fit(Xf, yf), {}
    rs = RandomizedSearchCV(
        clone(est), grid, n_iter=max_iter, cv=inner_folds,
        scoring="neg_mean_absolute_error", n_jobs=-1, random_state=42, refit=True
    )
    rs.fit(Xf, yf)
    return rs.best_estimator_, rs.best_params_

if grid:
    if BEST_MODEL == "RandomForest":
        # RF: OOB ist oft ausreichend → kein CV nötig
        final_model = clone(estimator).set_params(
            n_estimators=400, oob_score=True, bootstrap=True
        ).fit(X_final, y_arr)
        best_params = {"n_estimators": 400, "oob_score": True, "bootstrap": True}
        print("Best params (fast OOB):", best_params)
    else:
        final_model, best_params = small_random_search(estimator, grid, X_final, y_arr, inner_folds=3, n_iter=10)
        print("Best params (fast search):", best_params)
else:
    final_model = clone(estimator).fit(X_final, y_arr)
    best_params = {}

# --- 3) OOF-Evaluation schnell ---
def fast_oof(model, Xf, yf, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    oof = np.zeros(len(yf), dtype=float)
    for tr, te in kf.split(Xf, yf):
        m = clone(model)
        m.fit(Xf[tr], yf[tr])
        oof[te] = m.predict(Xf[te])
    return oof

if BEST_MODEL == "RandomForest" and getattr(final_model, "oob_score_", None) is not None:
    # OOB direkt nutzen (kein OOF nötig)
    oof_pred = final_model.oob_prediction_
else:
    oof_pred = fast_oof(final_model, X_final, y_arr, folds=5)  # 5-Fold, schnell

oof_mae  = mean_absolute_error(y_arr, oof_pred)
oof_rmse = np.sqrt(mean_squared_error(y_arr, oof_pred))
oof_r2   = r2_score(y_arr, oof_pred)

print(f"OOF MAE: {oof_mae:.4f} | OOF RMSE: {oof_rmse:.4f} | OOF R²: {oof_r2:.4f}")
print(f"Done in {time.time()-t0:.1f}s")


Best model: RandomForest | MAE nested: 1497.450962256624
Best params (fast OOB): {'n_estimators': 400, 'oob_score': True, 'bootstrap': True}
OOF MAE: 1429.3561 | OOF RMSE: 2501.5489 | OOF R²: 0.9340
Done in 39.8s


## Test-Prediction & Submission Export

In [30]:
# ==========================================
# Diagnose + Fix: Test-Predictions & Submission (32567 Zeilen)
# ==========================================
import numpy as np, pandas as pd, json, time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

REQ_ROWS = 32567
t0 = time.time()

def debug_shapes(label, df, Xt=None, Xf=None):
    print(f"[{label}] rows={len(df)}  | cols={df.shape[1]}")
    if Xt is not None: print(f"  transform: {Xt.shape}")
    if Xf is not None: print(f"  final    : {Xf.shape}")

# 0) Original-Test frisch laden (um Nebenwirkungen auszuschließen)
test_fresh = pd.read_csv(TEST_FILE)
print("Original TEST_FILE:", TEST_FILE)
debug_shapes("in-memory test (vorher)", test)
debug_shapes("fresh test.csv", test_fresh)

# Falls dein in-memory 'test' != original: wir verwenden das frische Original
if len(test_fresh) != REQ_ROWS:
    raise AssertionError(f"Das Original-Testfile hat {len(test_fresh)} Zeilen (erwartet {REQ_ROWS}). Bitte prüfen!")
test = test_fresh.copy()

# 1) Preprocess & Feature-Maske
_ = preprocessor.fit(X)
X_test_t = preprocessor.transform(test)

if "feature_names" not in globals():
    ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
    feature_names = [f"NUM::{c}" for c in num_cols] + [f"CAT::{n}" for n in ohe.get_feature_names_out(cat_cols)]

if "final" not in globals():
    SEL_FILE = Path("artifacts/selected_features.json")
    meta = json.loads(SEL_FILE.read_text(encoding="utf-8"))
    final = set(meta["final_features"])

idx_final = np.array([i for i, n in enumerate(feature_names) if n in final])
X_test_final = np.ascontiguousarray(X_test_t[:, idx_final])

debug_shapes("nach Prep", test, Xt=X_test_t, Xf=X_test_final)

# 2) Vorhersage
test_pred = final_model.predict(X_test_final)
print("pred len:", len(test_pred))

# 3) Safety: Falls trotz allem >32567 (z.B. wegen Duplikaten), per ID deduplizieren
id_col_candidates = [c for c in ["carID", "id", "ID"] if c in test.columns]
if len(test_pred) != REQ_ROWS:
    if id_col_candidates:
        idc = id_col_candidates[0]
        # Dedupliziere IDs stabil und bringe auf REQ_ROWS
        tmp = pd.DataFrame({idc: test[idc].values, TARGET: test_pred})
        before = tmp.shape[0]
        tmp = tmp.drop_duplicates(subset=[idc], keep="first")
        after = tmp.shape[0]
        print(f"⚠️ Dedup IDs: {before} → {after}")
        if after != REQ_ROWS:
            raise AssertionError(f"Nach Dedup hat die Testmenge {after} Zeilen (erwartet {REQ_ROWS}). Bitte Upstream prüfen!")
        sub = tmp.sort_values(idc).reset_index(drop=True)
    else:
        raise AssertionError(
            f"Vorhersage-Länge {len(test_pred)} ≠ {REQ_ROWS} und keine ID-Spalte zum Dedup gefunden. "
            "Bitte prüfen, ob 'test' unverändert aus TEST_FILE stammt."
        )
else:
    # 4) Normale Submission
    if id_col_candidates:
        idc = id_col_candidates[0]
        sub = pd.DataFrame({idc: test[idc].values, TARGET: test_pred})
        sub = sub.sort_values(idc).reset_index(drop=True)
    else:
        sub = pd.DataFrame({"ID": np.arange(len(test)), TARGET: test_pred})

# 5) Schreiben (CSV + ZIP)
SUBMISSIONS = Path("submissions"); SUBMISSIONS.mkdir(parents=True, exist_ok=True)
csv_path = SUBMISSIONS / "submission.csv"
sub.to_csv(csv_path, index=False)

zip_path = SUBMISSIONS / "submission.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname=csv_path.name)

print(f"✅ CSV: {csv_path.resolve()}")
print(f"🗜️ ZIP: {zip_path.resolve()}")
print(f"⏱️ Done in {time.time()-t0:.1f}s")
sub.head()


Original TEST_FILE: ../data/test.csv
[in-memory test (vorher)] rows=32567  | cols=13
[fresh test.csv] rows=32567  | cols=13
[nach Prep] rows=32567  | cols=13
  transform: (32567, 237)
  final    : (32567, 60)
pred len: 32567
✅ CSV: /Users/karaca/src/MachineLearningProject-NOVAIMS2025/notebooks/submissions/submission.csv
🗜️ ZIP: /Users/karaca/src/MachineLearningProject-NOVAIMS2025/notebooks/submissions/submission.zip
⏱️ Done in 2.9s


,carID,price
0,75973,10949.0375
1,75974,20300.9325
2,75975,26967.1275
3,75976,24284.9700
4,75977,16869.1400
